<font size=10>**PREPROCESSING**</font> <a class="anchor" id='title'></a> 

**Bachelor's in Data Science - NOVA IMS (25/26)**

**Project**: [*Seoul Bikes – Predicting Bike Rentals*](https://archive.ics.uci.edu/dataset/560/seoul+bike+sharing+demand)

**Group 8**
- Beatriz Marques 20231605
- David Carrilho 20231693
- Duarte Fernandes 20231619
- Filipe Caçador 20231707
- Mariana Calais-Pedro 20231641

*«notebook description»*

<font color='#BFD72' size=6>**TABLE OF CONTENTS**</font> <a class="anchor" id='toc'></a> 
- [1. Imports](#1)
- [2. Data Integration](#2)
- [3. Dataset Cleaning](#3)

# <font color='#BFD72F' size=6>**1. Imports**</font> <a class="anchor" id="1"></a>

[Back to TOC](#toc)

In [2]:
# Checking the installed Java version
!java -version
!pip install pyspark 

openjdk version "17.0.16" 2025-07-15
OpenJDK Runtime Environment (build 17.0.16+8-Ubuntu-0ubuntu124.04.1)
OpenJDK 64-Bit Server VM (build 17.0.16+8-Ubuntu-0ubuntu124.04.1, mixed mode, sharing)


In [3]:
# Install Java 17
!sudo apt-get update
!sudo apt-get install -y openjdk-17-jdk-headless

!java -version

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://download.docker.com/linux/ubuntu noble InRelease                 
Hit:3 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble InRelease          
Get:4 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Hit:5 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:6 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-security InRelease 
Hit:7 https://archive.ubuntu.com/ubuntu noble InRelease                        
Hit:8 https://archive.ubuntu.com/ubuntu noble-updates InRelease                
Hit:9 https://archive.ubuntu.com/ubuntu noble-backports InRelease              
Hit:10 https://packages.cloud.google.com/apt cloud-sdk InRelease               
Hit:11 https://us-east-2.ec2.archive.ubuntu.com/ubuntu noble InRelease         
Hit:12 https://us-east-2.ec2.archive.ubuntu.com/ubuntu noble-updates InRelease 
Hit:13 https://us-east-2.ec2.archive.ubuntu.com/ubuntu nob

In [4]:
# Set JAVA_HOME to Java 17
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession

spark = SparkSession.builder\
        .master("local[*]")\
        .appName("ML") \
        .getOrCreate()
        
print("Spark ready:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/08 13:11:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/11/08 13:11:35 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark ready: 4.0.1


In [ ]:
import warnings
%load_ext autoreload
%autoreload 2

warnings.filterwarnings('ignore')

In [5]:
import sys
import os

# Get the absolute path of the source_code folder
source_code_path = os.path.abspath('../../source')

# Add the source_code folder to sys.path
if source_code_path not in sys.path:
    sys.path.append(source_code_path)

from visualizations import *

In [6]:
# ===============================
# 📦 Core Libraries
# ===============================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

# ===============================
# 📊 Plotly Visualization
# ===============================
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ===============================
# ⚙️ PySpark Modules
# ===============================
from pyspark.sql import functions as F
from pyspark.sql.functions import col, coun
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from typing import List

# ===============================
# 🧩 Feature Engineering
# ===============================
from pyspark.ml.feature import (
    StringIndexer,       # Encodes categorical string values into numerical indices
    OneHotEncoder,       # Converts categorical variables into binary vectors
    VectorAssembler,     # Combines multiple feature columns into a single vector
    Normalizer           # Normalizes feature vectors
)

# ===============================
# 📈 Statistical Analysis
# ===============================
from pyspark.ml.stat import Correlation  # Computes correlation matrices for features

# ===============================
# 🤖 Machine Learning Models
# ===============================
from pyspark.ml.classification import (
    LogisticRegression,      # Logistic Regression model
    RandomForestClassifier,  # Random Forest classifier
    GBTClassifier             # Gradient-Boosted Trees classifier
)

# ===============================
# 🧮 Model Evaluation & Tuning
# ===============================
from pyspark.ml.evaluation import BinaryClassificationEvaluator  # Evaluates classification models
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator   # Hyperparameter tuning tools

# ===============================
# 📊 Metrics
# ===============================
from sklearn.metrics import roc_curve, auc  # ROC curve and AUC calculation

# <font color='#BFD72F' size=6>**2. Data Integration**</font> <a class="anchor" id="2"></a>
  
[Back to TOC](#toc)

In [7]:
data_path = "../../data/raw/SeoulBikeData.csv"

In [8]:
seoul_bikes_original = spark.read.load(data_path,
                                        format      = "csv",           
                                        sep         = ",",           
                                        inferSchema = "true",        
                                        header      = "true")

In [9]:
seoul_bikes_original

DataFrame[Date: string, Rented Bike Count: int, Hour: int, Temperature(�C): double, Humidity(%): int, Wind speed (m/s): double, Visibility (10m): int, Dew point temperature(�C): double, Solar Radiation (MJ/m2): double, Rainfall(mm): double, Snowfall (cm): double, Seasons: string, Holiday: string, Functioning Day: string]

In [10]:
# Making a copy to save the original file
seoul_bikes = seoul_bikes_original.alias('seoul_bikes')

In [11]:
seoul_bikes.show(5)

+----------+-----------------+----+---------------+-----------+----------------+----------------+-------------------------+-----------------------+------------+-------------+-------+----------+---------------+
|      Date|Rented Bike Count|Hour|Temperature(�C)|Humidity(%)|Wind speed (m/s)|Visibility (10m)|Dew point temperature(�C)|Solar Radiation (MJ/m2)|Rainfall(mm)|Snowfall (cm)|Seasons|   Holiday|Functioning Day|
+----------+-----------------+----+---------------+-----------+----------------+----------------+-------------------------+-----------------------+------------+-------------+-------+----------+---------------+
|01/12/2017|              254|   0|           -5.2|         37|             2.2|            2000|                    -17.6|                    0.0|         0.0|          0.0| Winter|No Holiday|            Yes|
|01/12/2017|              204|   1|           -5.5|         38|             0.8|            2000|                    -17.6|                    0.0|         0.0|

In [12]:
# RENAMING COLUMNS TO EASIER ACCESS
seoul_bikes = seoul_bikes.select(
    [F.col(c).alias(c.replace(" ", "_")
                  .replace("(", "")
                  .replace(")", "")
                  .replace("�", "_")
                  .replace("/", "")
                  .replace("%", "_Percentage")) for c in seoul_bikes.columns]
)

In [13]:
# CONVER DATE TO A DATATIME FORMAT
seoul_bikes = seoul_bikes.withColumn("Date", F.to_date(F.col("Date"), "dd/MM/yyyy"))

**Target Column:** *Rented Bike Count*

## <font color='#BFD72F' size=6>2.1 Data Partition</font> <a class="anchor" id="2_!"></a>

[Back to TOC](#toc)

In [17]:
# Sort the dataset by date
seoul_bikes_sorted = seoul_bikes.orderBy('Date')

# Count total rows
total_rows = seoul_bikes_sorted.count()
train_size = int(total_rows * 0.8)

# Split chronologically
train_df = seoul_bikes_sorted.limit(train_size)
val_df = seoul_bikes_sorted.subtract(train_df)

# Training sets
X_train = train_df.drop('Rented_Bike_Count')
y_train = train_df.select('Rented_Bike_Count')

# Validation sets
X_val = val_df.drop('Rented_Bike_Count')
y_val = val_df.select('Rented_Bike_Count')

# <font color='#BFD72F' size=6>**3. Dataset Cleaning**</font> <a class="anchor" id="P3"></a>

[Back to TOC](#toc)

## <font color='#BFD72F' size=6>3.1  Drop Unmeaningful Data</font> <a class="anchor" id="3_1"></a>

[Back to TOC](#toc)

## <font color='#BFD72F' size=6>3.2  Treating Outliers</font> <a class="anchor" id="3_2"></a>

[Back to TOC](#toc)

In [14]:
def winsorize_spark(reference_df: DataFrame, apply_to_df: DataFrame, columns: List[str]) -> DataFrame:
    """
    Apply winsorization to specified columns in a Spark DataFrame using IQR from a reference DataFrame.

    Parameters:
        reference_df (DataFrame): Spark DataFrame used to calculate IQR.
        apply_to_df (DataFrame): Spark DataFrame to apply winsorization.
        columns (List[str]): List of column names to winsorize.

    Returns:
        DataFrame: Spark DataFrame with winsorized columns.
    """
    for col in columns:
        # Calculate Q1 and Q3 from reference_df
        q1, q3 = reference_df.approxQuantile(col, [0.25, 0.75], 0.01)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        # Apply winsorization using when/otherwise
        apply_to_df = apply_to_df.withColumn(
            col,
            F.when(F.col(col) < lower_bound, lower_bound)
             .when(F.col(col) > upper_bound, upper_bound)
             .otherwise(F.col(col))
        )

    return apply_to_df

In [15]:
numerical_cols = [
    'Temperature_C',
    'Humidity_Percentage',
    'Wind_speed_ms',
    'Visibility_10m',
    'Dew_point_temperature_C',
    'Solar_Radiation_MJm2',
    'Rainfallmm',
    'Snowfall_cm',
]

In [18]:
X_train = winsorize_spark(reference_df=X_train, apply_to_df=X_train, columns=numerical_cols)
X_val = winsorize_spark(reference_df=X_train, apply_to_df=X_val, columns=numerical_cols)

In [19]:
box_plots_spark(
    X_train,
    X_val,
    numerical_cols
)

## <font color='#BFD72F' size=6>3.3  Feature Engineering</font> <a class="anchor" id="3_3"></a>

[Back to TOC](#toc)

In [22]:
def feature_engineering_spark(df: DataFrame) -> DataFrame:
    """
    Perform feature engineering on a Spark DataFrame.
    Parameters:
        df (DataFrame): Spark DataFrame.

    Returns:
        DataFrame: Spark DataFrame with the new columns added.
    """

    # df = df.withColumn("bmi", F.col("weight") / (F.col("height") ** 2))

    return df

In [24]:
X_train = feature_engineering_spark(X_train)
X_val = feature_engineering_spark(X_val)

## <font color='#BFD72F' size=6>3.4  Encoding</font> <a class="anchor" id="3_4"></a>

[Back to TOC](#toc)

## <font color='#BFD72F' size=6>3.5  Scaling</font> <a class="anchor" id="3_5"></a>

[Back to TOC](#toc)

## <font color='#BFD72F' size=6>3.6  Treating Missing Values</font> <a class="anchor" id="3_6"></a>

[Back to TOC](#toc)

## <font color='#BFD72F' size=6>3.7  Feature Selection</font> <a class="anchor" id="3_7"></a>

[Back to TOC](#toc)